## 02 — Chunk documents
Splits every document into chunks (header + piece of body) and keeps metadata
for filtering. The chunks table is the source of the AI Search index.

In [0]:
for key in ["catalog", "silver_schema", "gold_schema", "rag_schema"]:
    dbutils.widgets.text(key, "")

In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

from rag_youtube.chunking import build_chunks

chunk_size = config["chunking"]["chunk_size"]
chunk_overlap = config["chunking"]["chunk_overlap"]


def chunk_document(header, body):
    return build_chunks(header, body, chunk_size, chunk_overlap)


build_chunks_udf = F.udf(chunk_document, ArrayType(StringType()))

In [0]:
def labeled(label, column):
    return F.when(F.col(column) != "", F.concat(F.lit(label), F.col(column)))


chunks_df = (
    spark.table(documents_table)
    .withColumn(
        "header",
        F.concat_ws("\n",
                    labeled("Title: ", "title"),
                    labeled("Channel: ", "channel_title"),
                    labeled("Category: ", "category_name")),
    )
    .withColumn(
        "body",
        F.concat_ws("\n",
                    labeled("Tags: ", "tags"),
                    labeled("Description: ", "description")),
    )
    .select(
        "video_id",
        "title",
        "channel_title",
        "category_name",
        "last_trending_date",
        "views",
        F.posexplode(build_chunks_udf("header", "body")).alias("chunk_index", "chunk_text"),
    )
    .withColumn("chunk_id", F.concat_ws("_", F.col("video_id"), F.col("chunk_index").cast("string")))
)

In [0]:
stats = chunks_df.agg(
    F.count("*").alias("chunks"),
    F.countDistinct("video_id").alias("videos"),
    F.round(F.avg(F.length("chunk_text"))).alias("avg_len"),
    F.max(F.length("chunk_text")).alias("max_len"),
).first()

multi_chunk_videos = chunks_df.filter(F.col("chunk_index") > 0).select("video_id").distinct().count()

logger.info(
    "Chunks: %s from %s videos (%s videos split into several chunks), avg length %s, max length %s",
    stats["chunks"], stats["videos"], multi_chunk_videos, stats["avg_len"], stats["max_len"],
)

In [0]:
ai_cfg = config["ai_search"]
embedding_col = ai_cfg["embedding_column"]
embedding_endpoint = ai_cfg["embedding_endpoint"]

if spark.catalog.tableExists(chunks_table):
    existing_df = spark.table(chunks_table).select(
        "chunk_id",
        F.col("chunk_text").alias("existing_text"),
        embedding_col,
    )
    compared_df = chunks_df.join(existing_df, on="chunk_id", how="left")

    reused_df = (
        compared_df
        .filter(F.col("existing_text") == F.col("chunk_text"))
        .drop("existing_text")
    )
    to_embed_df = (
        compared_df
        .filter(F.col("existing_text").isNull() | (F.col("existing_text") != F.col("chunk_text")))
        .drop("existing_text", embedding_col)
    )
else:
    reused_df = None
    to_embed_df = chunks_df

embedded_df = to_embed_df.withColumn(
    embedding_col,
    F.expr(f"ai_query('{embedding_endpoint}', chunk_text)").cast("array<float>"),
)

source_df = embedded_df if reused_df is None else reused_df.unionByName(embedded_df)

In [0]:
to_embed_count = to_embed_df.count()
logger.info("Chunks to embed: %s (reused: %s)", to_embed_count, chunks_df.count() - to_embed_count)

(
    source_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(chunks_staging_table)
)

check = spark.table(chunks_staging_table).agg(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col(embedding_col).isNull(), 1).otherwise(0)).alias("null_vectors"),
    F.sum(F.when(F.size(embedding_col) != ai_cfg["embedding_dimension"], 1).otherwise(0)).alias("wrong_size"),
).first()

logger.info("Staging %s: rows=%s, null vectors=%s, wrong size=%s",
            chunks_staging_table, check["rows"], check["null_vectors"], check["wrong_size"])

if check["null_vectors"] or check["wrong_size"]:
    raise ValueError("Some chunks have missing or wrong-size embeddings, stopping before MERGE")

In [0]:
from delta.tables import DeltaTable

staging_df = spark.table(chunks_staging_table)

(
    DeltaTable.createIfNotExists(spark)
    .tableName(chunks_table)
    .addColumns(staging_df.schema)
    .property("delta.enableChangeDataFeed", "true")
    .comment("Lab 12: chunks of YouTube video documents with embeddings, source of the AI Search index")
    .execute()
)

In [0]:
target = DeltaTable.forName(spark, chunks_table)

changed = """
NOT (
    t.chunk_text <=> s.chunk_text
    AND t.title <=> s.title
    AND t.channel_title <=> s.channel_title
    AND t.category_name <=> s.category_name
    AND t.last_trending_date <=> s.last_trending_date
    AND t.views <=> s.views
)
"""

(
    target.alias("t")
    .merge(staging_df.alias("s"), "t.chunk_id = s.chunk_id")
    .whenMatchedUpdateAll(condition=changed)
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)

In [0]:
last_merge = target.history(1).select("operationMetrics").first()[0]

logger.info(
    "Chunks table %s: inserted=%s, updated=%s, deleted=%s, total=%s",
    chunks_table,
    last_merge.get("numTargetRowsInserted"),
    last_merge.get("numTargetRowsUpdated"),
    last_merge.get("numTargetRowsDeleted"),
    spark.table(chunks_table).count(),
)